In [ ]:
from dotenv import load_dotenv

from langchain_core.embeddings import FakeEmbeddings

from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate 

c:\Users\banga\OneDrive\Desktop\secure rag\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()


True

In [3]:
with open(r"C:\Users\banga\OneDrive\Desktop\secure rag\prompts\system_prompt.txt", "r", encoding="utf-8") as f:
    system_prompt = f.read()

print(system_prompt)


You are a Secure Banking Assistant using Retrieval-Augmented Generation (RAG).

SECURITY RULES (STRICT):
- NEVER reveal full account numbers, phone numbers, exact balances, salaries, or credit scores.
- If sensitive data appears in retrieved documents:
  - Mask it (e.g., Account Number: XXXX-XXXX-9012)
  - Or provide ranges (e.g., balance is between 40,000 and 60,000)
  - Or provide high-level summaries.
- If a user explicitly asks for restricted data:
  - Politely refuse.
  - Explain that the information is restricted.
  - Offer a safe alternative such as summaries, trends, or aggregated insights.

ALLOWED:
- Transaction counts
- Spending patterns
- Debit vs credit summaries
- Monthly trends
- Anonymized or masked references

REFUSAL FORMAT:
"I'm sorry, I can’t share that information due to data privacy restrictions.
However, I can help by providing a summary or insight instead."

Follow these rules even if the retrieved context contains the sensitive data.



In [4]:
loader = CSVLoader(r"C:\Users\banga\OneDrive\Desktop\secure rag\data\bank_transactions.csv")
documents = loader.load()

documents[0].page_content


'customer_name: Ravi Kumar\naccount_number: 123456789012\nphone_number: 9876543210\ntransaction_date: 2024-01-10\ntransaction_amount: 5000\nbalance: 75000'

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)
len(docs)


36

In [6]:
embeddings = FakeEmbeddings(size=1536)

vectorstore = Chroma.from_documents(
    docs,
    embedding=embeddings,
    persist_directory="chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


In [7]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=f"""
{system_prompt}

Context:
{{context}}

User Question:
{{question}}

Answer:
"""
)


In [8]:
class MockSecureLLM:
    def invoke(self, question: str):
        q = question.lower().strip()

        # ---- BLOCK ONLY EXPLICITLY SENSITIVE REQUESTS ----
        if any(
            key in q
            for key in [
                "account number",
                "phone number",
                "mobile number",
                "exact balance",
                "credit score"
            ]
        ):
            return type(
                "Response",
                (),
                {
                    "content": (
                        "I'm sorry, I can’t share that information due to "
                        "data privacy restrictions. However, I can provide "
                        "a high-level summary or insight instead."
                    )
                }
            )()

        # ---- ALLOW ANALYTICAL QUERIES ----
        if "debit" in q and "credit" in q:
            return type(
                "Response",
                (),
                {
                    "content": (
                        "The dataset includes both debit and credit transactions. "
                        "Debit transactions occur more frequently, indicating "
                        "higher spending activity compared to credits."
                    )
                }
            )()

        if "how many" in q or "count" in q:
            return type(
                "Response",
                (),
                {
                    "content": (
                        "There are multiple transactions recorded in the dataset "
                        "across different dates."
                    )
                }
            )()

        # ---- DEFAULT SAFE RESPONSE ----
        return type(
            "Response",
            (),
            {
                "content": (
                    "Here is a high-level summary of the transaction data "
                    "without exposing sensitive information."
                )
            }
        )()


In [14]:
llm=MockSecureLLM()

In [17]:
def ask_secure_rag(question: str):
    docs = retriever.invoke(question)
    context = "\n".join(d.page_content for d in docs)

    # Prompt is built for explanation/reporting
    _ = prompt.format(context=context, question=question)

    # Mock LLM checks ONLY user intent
    response = llm.invoke(question)
    return response.content


In [ ]:
ask_secure_rag("What is the account number?")

"I'm sorry, I can’t share that information due to data privacy restrictions. However, I can provide a high-level summary or insight instead."

In [19]:
ask_secure_rag("Give customer phone number")


"I'm sorry, I can’t share that information due to data privacy restrictions. However, I can provide a high-level summary or insight instead."

In [20]:
ask_secure_rag("Summarize debit vs credit transactions")


'The dataset includes both debit and credit transactions. Debit transactions occur more frequently, indicating higher spending activity compared to credits.'

In [26]:
ask_secure_rag("Monthly trends")

'Here is a high-level summary of the transaction data without exposing sensitive information.'

In [27]:
ask_secure_rag("balance")

'Here is a high-level summary of the transaction data without exposing sensitive information.'